## 4-1-3 STACKED MODEL GPR,XGB,MLP

In [ ]:
# ==========================================================
# CELL 1: IMPORT REQUIRED LIBRARIES
# ==========================================================
# SP_GR HYBRID STACKING
# RF + XGB + ANN
# ==========================================================

import numpy as np
import pandas as pd
import random
import joblib
import tensorflow as tf

from sklearn.model_selection import GroupKFold

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping

# ==========================================================
# RANDOM SEED
# ==========================================================

SEED = 42

np.random.seed(SEED)

random.seed(SEED)

tf.random.set_seed(SEED)

print("Libraries Loaded Successfully")

# ==========================================================
# CELL 2: LOAD SP_GR DATASET
# ==========================================================

df = pd.read_csv(
    "sp_gr_dataset_clean.csv"
)

TARGET = "Hydrate Temperature (K)"

# ==========================================================
# FEATURES
# ==========================================================

X = df[
    [
        "Specific Gravity",
        "Pressure (psia)"
    ]
]

y = df[TARGET]

# ==========================================================
# GROUPING
# SAME GAS AT DIFFERENT PRESSURES
# ==========================================================

df["group_id"] = (
    df["Specific Gravity"]
    .round(4)
    .astype(str)
)

groups = df["group_id"]

# ==========================================================
# CELL 3: TRAIN TEST SPLIT
# ==========================================================

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups
    )
)

X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

print("Train Samples :", len(X_train))
print("Test Samples  :", len(X_test))

print(
    "Train Groups :",
    groups_train.nunique()
)

print(
    "Test Groups :",
    groups.iloc[test_idx].nunique()
)

Libraries Loaded Successfully
Train Samples : 3076
Test Samples  : 735
Train Groups : 1581
Test Groups : 396


In [64]:
# ==========================================================
# CELL 4: DEFINE BASE LEARNERS
# ==========================================================

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

from xgboost import XGBRegressor

# ==========================================================
# GPR
# ==========================================================

kernel = (
    ConstantKernel(1.0)
    *
    Matern(
        length_scale=0.5,
        nu=1.5
    )
    +
    WhiteKernel(
        noise_level=1e-5
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,
    normalize_y=True,
    random_state=42
)

# ==========================================================
# XGB
# ==========================================================

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

# ==========================================================
# MLP
# ==========================================================

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=2000,
    random_state=42
)

print("GPR, XGB and MLP Initialized")

GPR, XGB and MLP Initialized


In [65]:
# ==========================================================
# CELL 6: INITIALIZE GROUPKFOLD AND OOF ARRAYS
# ==========================================================
# Out-of-Fold predictions will be generated using
# GroupKFold to avoid data leakage.
# ==========================================================

from sklearn.model_selection import GroupKFold

gkf = GroupKFold(
    n_splits=5
)

# ==========================================================
# OOF ARRAYS
# ==========================================================

oof_gpr = np.zeros(
    len(X_train)
)

oof_xgb = np.zeros(
    len(X_train)
)

oof_ann = np.zeros(
    len(X_train)
)

print(
    "Number of Training Samples:",
    len(X_train)
)

print(
    "Number of Groups:",
    groups_train.nunique()
)

print(
    "Number of Folds:",
    gkf.get_n_splits()
)

print(
    "OOF Arrays Created Successfully"
)

Number of Training Samples: 3076
Number of Groups: 1581
Number of Folds: 5
OOF Arrays Created Successfully


In [66]:
# ==========================================================
# CELL 6: INITIALIZE OOF ARRAYS
# ==========================================================

oof_gpr = np.zeros(len(X_train))

oof_xgb = np.zeros(len(X_train))

oof_mlp = np.zeros(len(X_train))

print("OOF Arrays Created")
print("Length =", len(oof_gpr))

OOF Arrays Created
Length = 3076


In [67]:
# ==========================================================
# CELL 7: GENERATE OOF PREDICTIONS
# ==========================================================
from sklearn.cluster import KMeans
from sklearn.gaussian_process.kernels import ConstantKernel as C
from sklearn.gaussian_process.kernels import (
    Matern,
    WhiteKernel
)
for fold, (train_idx, valid_idx) in enumerate(

    gkf.split(
        X_train,
        y_train,
        groups_train
    )

):

    print("\n" + "="*60)
    print(f"FOLD {fold+1}")
    print("="*60)

    # ======================================================
    # SPLIT
    # ======================================================

    X_tr = X_train.iloc[train_idx]
    X_va = X_train.iloc[valid_idx]

    y_tr = y_train.iloc[train_idx]
    y_va = y_train.iloc[valid_idx]

    # ======================================================
    # RF
    # ======================================================

    gpr.fit(
        X_tr,
        y_tr.values
    )

    gpr_pred = gpr.predict(
        X_va
    )

    oof_gpr[valid_idx] = gpr_pred

    # ======================================================
    # XGB
    # ======================================================

    xgb.fit(
        X_tr,
        y_tr
    )

    xgb_pred = xgb.predict(
        X_va
    )

    oof_xgb[valid_idx] = xgb_pred

    # ======================================================
    # MLP SCALING
    # ======================================================

    X_scaler = StandardScaler()

    X_tr_mlp = X_scaler.fit_transform(
        X_tr
    )

    X_va_mlp = X_scaler.transform(
        X_va
    )

    # ======================================================
    # MLP
    # ======================================================

    mlp.fit(
        X_tr_mlp,
        y_tr
    )

    mlp_pred = mlp.predict(
        X_va_mlp
    )

    oof_mlp[valid_idx] = mlp_pred

    # ======================================================
    # METRICS
    # ======================================================

    print(
        "RF R2 =",
        round(
            r2_score(
                y_va,
                gpr_pred
            ),
            6
        )
    )

    print(
        "XGB R2 =",
        round(
            r2_score(
                y_va,
                xgb_pred
            ),
            6
        )
    )

    print(
        "MLP R2 =",
        round(
            r2_score(
                y_va,
                mlp_pred
            ),
            6
        )
    )


FOLD 1


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


RF R2 = 0.966785
XGB R2 = 0.973448
MLP R2 = 0.964817

FOLD 2


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\_gpr.py:629: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


RF R2 = 0.936829
XGB R2 = 0.945221
MLP R2 = 0.961183

FOLD 3


...

RF R2 = 0.970634
XGB R2 = 0.970838
MLP R2 = 0.964713

FOLD 4


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\_gpr.py:629: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


RF R2 = 0.97882
XGB R2 = 0.980675
MLP R2 = 0.978736

FOLD 5


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


RF R2 = 0.978519
XGB R2 = 0.976803
MLP R2 = 0.963145


In [69]:
# ==========================================================
# TRAIN BASE MODELS
# GPR + XGB + MLP
# ==========================================================

from sklearn.preprocessing import StandardScaler
import joblib

# ==========================================================
# RETRAIN GPR ON FULL TRAINING DATA
# ==========================================================

gpr.fit(
    X_train,
    y_train
)

# ==========================================================
# RETRAIN XGB ON FULL TRAINING DATA
# ==========================================================

xgb.fit(
    X_train,
    y_train
)

# ==========================================================
# RETRAIN MLP ON FULL TRAINING DATA
# ==========================================================

X_scaler = StandardScaler()

X_train_mlp = X_scaler.fit_transform(
    X_train
)

mlp.fit(
    X_train_mlp,
    y_train
)

print("Base Models Trained Successfully")

# ==========================================================
# SAVE BASE MODELS
# ==========================================================

joblib.dump(
    gpr,
    "GPR118_base_model.pkl"
)

joblib.dump(
    xgb,
    "XGB118_base_model.pkl"
)

joblib.dump(
    mlp,
    "MLP118_base_model.pkl"
)

joblib.dump(
    X_scaler,
    "MLP118_scaler.pkl"
)

print("Base Models Saved Successfully")

C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Base Models Trained Successfully
Base Models Saved Successfully


# START

In [83]:
# ==========================================================
# TRAIN META MODELS
# LINEAR + XGB
# ==========================================================

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import pandas as pd
import joblib

# ==========================================================
# CREATE META DATASET
# ==========================================================

meta_X = pd.DataFrame({

    "GPR": oof_gpr,

    #"XGB": oof_xgb,

    "MLP": oof_mlp


})

meta_y = y_train.copy()

print(meta_X.head())
print(meta_X.shape)

# ==========================================================
# TRAIN LINEAR META
# ==========================================================

linear_meta = LinearRegression()

linear_meta.fit(
    meta_X,
    meta_y
)

print("Linear Meta Trained")

# ==========================================================
# TRAIN XGB META
# ==========================================================

xgb_meta = XGBRegressor(
    n_estimators=600,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_meta.fit(
    meta_X,
    meta_y
)

print("XGB Meta Trained")

# ==========================================================
# SAVE META MODELS
# ==========================================================

joblib.dump(
    linear_meta,
    "Linear8_meta_model_WITHOUT XGB.pkl"
)

joblib.dump(
    xgb_meta,
    "XGB8_meta_model_WITHOUT XGB.pkl"
)

print("Meta Models Saved Successfully")

          GPR         MLP
0  295.567570  297.078490
1  293.036549  293.785288
2  291.857468  292.408152
3  290.472705  289.419855
4  286.898429  283.362861
(3076, 2)
Linear Meta Trained
XGB Meta Trained
Meta Models Saved Successfully


In [84]:
# ==========================================================
# CELL 1: IMPORT LIBRARIES
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
# ==========================================================
# CELL 2: LOAD MODELS
# ==========================================================

gpr_model = joblib.load(
    "GPR118_base_model.pkl"
)

xgb_model = joblib.load(
    "XGB118_base_model.pkl"
)

mlp_model = joblib.load(
    "MLP118_base_model.pkl"
)

MLP_X_scaler = joblib.load(
    "MLP118_scaler.pkl"
)


linear_meta = joblib.load(
    "Linear8_meta_model_WITHOUT XGB.pkl"
)

xgb_meta = joblib.load(
    "XGB8_meta_model_WITHOUT XGB.pkl"
)

# ==========================================================
# CELL 3: INPUT DATA
# ==========================================================

SP_GR = 0.643  # levan

pressure = np.array([
    690.00,
    878.33,
    1066.67,
    1255.00,
    1443.33,
    1631.67,
    1820.00,
    2008.33,
    2196.67,
    2385.00,
    2573.33,
    2761.67,
    2950.00
])

experimental = np.array([
    285.90,
    287.32,
    288.52,
    289.64,
    290.60,
    291.37,
    291.98,
    292.48,
    292.90,
    293.30,
    293.69,
    294.07,
    294.45
])

# ==========================================================
# CELL 4: CREATE INPUT DATASET
# ==========================================================

X_new = pd.DataFrame({

    "Specific Gravity": [SP_GR] * len(pressure),

    "Pressure (psia)": pressure

})

# ==========================================================
# CELL 5: BASE MODEL PREDICTIONS
# ==========================================================

gpr_pred = gpr_model.predict(
    X_new
)

xgb_pred = xgb_model.predict(
    X_new
)

X_mlp = MLP_X_scaler.transform(
    X_new
)

mlp_pred = mlp_model.predict(
    X_mlp
)

if len(mlp_pred.shape) > 1:
    mlp_pred = mlp_pred.ravel()

# ==========================================================
# CELL 6: CREATE META FEATURES
# ==========================================================

meta_features = pd.DataFrame({

    
    "GPR": gpr_pred,

    #"XGB": xgb_pred,

    "MLP": mlp_pred

})

# ==========================================================
# CELL 7: META MODEL PREDICTIONS
# ==========================================================

hybrid_linear_pred = linear_meta.predict(
    meta_features
)

hybrid_xgb_pred = xgb_meta.predict(
    meta_features
)

# ==========================================================
# CELL 8: METRICS FUNCTION
# ==========================================================

def calculate_metrics(y_true, y_pred):

    r2 = r2_score(
        y_true,
        y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    rrmse = (
        rmse /
        np.mean(y_true)
    ) * 100

    return r2, mae, rmse, rrmse

# ==========================================================
# CELL 9: EXTERNAL VALIDATION
# ==========================================================

gpr_metrics = calculate_metrics(
    experimental,
    gpr_pred
)

xgb_metrics = calculate_metrics(
    experimental,
    xgb_pred
)

mlp_metrics = calculate_metrics(
    experimental,
    mlp_pred
)

linear_metrics = calculate_metrics(
    experimental,
    hybrid_linear_pred
)

xgbmeta_metrics = calculate_metrics(
    experimental,
    hybrid_xgb_pred
)

metrics = pd.DataFrame({

    "Model": [
        "GPR",
        "XGB",
        "MLP",
        "Hybrid_Linear_Meta",
        "Hybrid_XGB_Meta"
    ],

    "R2": [
        gpr_metrics[0],
        xgb_metrics[0],
        mlp_metrics[0],
        linear_metrics[0],
        xgbmeta_metrics[0]
    ],

    "MAE": [
        gpr_metrics[1],
        xgb_metrics[1],
        mlp_metrics[1],
        linear_metrics[1],
        xgbmeta_metrics[1]
    ],

    "RMSE": [
        gpr_metrics[2],
        xgb_metrics[2],
        mlp_metrics[2],
        linear_metrics[2],
        xgbmeta_metrics[2]
    ],

    "RRMSE (%)": [
        gpr_metrics[3],
        xgb_metrics[3],
        mlp_metrics[3],
        linear_metrics[3],
        xgbmeta_metrics[3]
    ]

})

print("\nExternal Validation Results\n")

print(
    metrics.sort_values(
        by="R2",
        ascending=False
    )
)

# ==========================================================
# CELL 10: PREDICTIONS TABLE
# ==========================================================

results = pd.DataFrame({

    "Pressure (psia)": pressure,

    "Experimental": experimental,

    "GPR_Pred": gpr_pred,

    "XGB_Pred": xgb_pred,

    "MLP_Pred": mlp_pred,

    "Hybrid_Linear_Pred": hybrid_linear_pred,

    "Hybrid_XGB_Pred": hybrid_xgb_pred

})

print(results)

results.to_excel(
    "SPGR_Predictions.xlsx",
    index=False
)

print("\nResults Saved Successfully")


External Validation Results

                Model        R2       MAE      RMSE  RRMSE (%)
2                 MLP  0.998772  0.069537  0.090988   0.031241
1                 XGB  0.986510  0.250365  0.301615   0.103560
4     Hybrid_XGB_Meta  0.981791  0.290360  0.350422   0.120318
3  Hybrid_Linear_Meta  0.977606  0.352055  0.388604   0.133427
0                 GPR  0.923345  0.647214  0.718974   0.246860
    Pressure (psia)  Experimental    GPR_Pred    XGB_Pred    MLP_Pred  \
0            690.00        285.90  286.703095  285.388855  286.125861   
1            878.33        287.32  288.613286  287.453979  287.478927   
2           1066.67        288.52  289.442643  288.866974  288.607242   
3           1255.00        289.64  290.383980  290.044159  289.732229   
4           1443.33        290.60  290.984783  290.628693  290.657369   
5           1631.67        291.37  292.185899  291.287964  291.323253   
6           1820.00        291.98  292.457537  291.664276  291.957739   
7       

In [76]:
# ==========================================================
# CELL 1: IMPORT LIBRARIES
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
# ==========================================================
# CELL 2: LOAD MODELS
# ==========================================================

gpr_model = joblib.load(
    "GPR118_base_model.pkl"
)

xgb_model = joblib.load(
    "XGB118_base_model.pkl"
)

mlp_model = joblib.load(
    "MLP118_base_model.pkl"
)

MLP_X_scaler = joblib.load(
    "MLP118_scaler.pkl"
)


linear_meta = joblib.load(
    "Linear8_meta_model_WITHOUT MLP.pkl"
)

xgb_meta = joblib.load(
    "XGB8_meta_model_WITHOUT MLP.pkl"
)

# ==========================================================
# CELL 3: INPUT DATA
# ==========================================================

SP_GR = 0.799  # SAL

# ==========================================================
# PRESSURE
# ==========================================================

pressure = np.array([
     700.00,
     891.67,
    1083.33,
    1275.00,
    1466.67,
    1658.33,
    1850.00,
    2041.67,
    2233.33,
    2425.00,
    2616.67,
    2808.33,
    3000.00
])

# ==========================================================
# EXPERIMENTAL
# ==========================================================

experimental = np.array([
    289.86,
    291.93,
    293.02,
    293.68,
    294.26,
    294.79,
    295.26,
    295.68,
    296.07,
    296.44,
    296.79,
    297.13,
    297.49
])


# ==========================================================
# CELL 4: CREATE INPUT DATASET
# ==========================================================

X_new = pd.DataFrame({

    "Specific Gravity": [SP_GR] * len(pressure),

    "Pressure (psia)": pressure

})

# ==========================================================
# CELL 5: BASE MODEL PREDICTIONS
# ==========================================================

gpr_pred = gpr_model.predict(
    X_new
)

xgb_pred = xgb_model.predict(
    X_new
)

X_mlp = MLP_X_scaler.transform(
    X_new
)

mlp_pred = mlp_model.predict(
    X_mlp
)

if len(mlp_pred.shape) > 1:
    mlp_pred = mlp_pred.ravel()

# ==========================================================
# CELL 6: CREATE META FEATURES
# ==========================================================

meta_features = pd.DataFrame({

    "GPR": gpr_pred,

    "XGB": xgb_pred,

    #"MLP": mlp_pred

})

# ==========================================================
# CELL 7: META MODEL PREDICTIONS
# ==========================================================

hybrid_linear_pred = linear_meta.predict(
    meta_features
)

hybrid_xgb_pred = xgb_meta.predict(
    meta_features
)

# ==========================================================
# CELL 8: METRICS FUNCTION
# ==========================================================

def calculate_metrics(y_true, y_pred):

    r2 = r2_score(
        y_true,
        y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    rrmse = (
        rmse /
        np.mean(y_true)
    ) * 100

    return r2, mae, rmse, rrmse

# ==========================================================
# CELL 9: EXTERNAL VALIDATION
# ==========================================================

gpr_metrics = calculate_metrics(
    experimental,
    gpr_pred
)

xgb_metrics = calculate_metrics(
    experimental,
    xgb_pred
)

mlp_metrics = calculate_metrics(
    experimental,
    mlp_pred
)

linear_metrics = calculate_metrics(
    experimental,
    hybrid_linear_pred
)

xgbmeta_metrics = calculate_metrics(
    experimental,
    hybrid_xgb_pred
)

metrics = pd.DataFrame({

    "Model": [
        "GPR",
        "XGB",
        "MLP",
        "Hybrid_Linear_Meta",
        "Hybrid_XGB_Meta"
    ],

    "R2": [
        gpr_metrics[0],
        xgb_metrics[0],
        mlp_metrics[0],
        linear_metrics[0],
        xgbmeta_metrics[0]
    ],

    "MAE": [
        gpr_metrics[1],
        xgb_metrics[1],
        mlp_metrics[1],
        linear_metrics[1],
        xgbmeta_metrics[1]
    ],

    "RMSE": [
        gpr_metrics[2],
        xgb_metrics[2],
        mlp_metrics[2],
        linear_metrics[2],
        xgbmeta_metrics[2]
    ],

    "RRMSE (%)": [
        gpr_metrics[3],
        xgb_metrics[3],
        mlp_metrics[3],
        linear_metrics[3],
        xgbmeta_metrics[3]
    ]

})

print("\nExternal Validation Results\n")

print(
    metrics.sort_values(
        by="R2",
        ascending=False
    )
)

# ==========================================================
# CELL 10: PREDICTIONS TABLE
# ==========================================================

results = pd.DataFrame({

    "Pressure (psia)": pressure,

    "Experimental": experimental,

    "GPR_Pred": gpr_pred,

    "XGB_Pred": xgb_pred,

    "MLP_Pred": mlp_pred,

    "Hybrid_Linear_Pred": hybrid_linear_pred,

    "Hybrid_XGB_Pred": hybrid_xgb_pred

})

print(results)

results.to_excel(
    "SPGR_Predictions.xlsx",
    index=False
)

print("\nResults Saved Successfully")


External Validation Results

                Model        R2       MAE      RMSE  RRMSE (%)
1                 XGB  0.906729  0.499386  0.652101   0.221201
3  Hybrid_Linear_Meta  0.792405  0.811159  0.972858   0.330006
4     Hybrid_XGB_Meta  0.743762  0.942656  1.080842   0.366636
2                 MLP  0.471668  1.367572  1.552008   0.526461
0                 GPR  0.460201  1.291951  1.568761   0.532144
    Pressure (psia)  Experimental    GPR_Pred    XGB_Pred    MLP_Pred  \
0            700.00        289.86  288.108238  288.765076  287.282001   
1            891.67        291.93  288.898989  290.604706  289.532251   
2           1083.33        293.02  290.802890  292.243195  290.681846   
3           1275.00        293.68  291.775571  293.453918  291.804332   
4           1466.67        294.26  292.594685  294.004913  292.749624   
5           1658.33        294.79  293.267747  294.664185  293.395767   
6           1850.00        295.26  293.704693  295.040497  293.844743   
7       

In [77]:
# ==========================================================
# CELL 1: IMPORT LIBRARIES
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
# ==========================================================
# CELL 2: LOAD MODELS
# ==========================================================

gpr_model = joblib.load(
    "GPR118_base_model.pkl"
)

xgb_model = joblib.load(
    "XGB118_base_model.pkl"
)

mlp_model = joblib.load(
    "MLP118_base_model.pkl"
)

MLP_X_scaler = joblib.load(
    "MLP118_scaler.pkl"
)


linear_meta = joblib.load(
    "Linear8_meta_model_WITHOUT MLP.pkl"
)

xgb_meta = joblib.load(
    "XGB8_meta_model_WITHOUT MLP.pkl"
)
# ==========================================================
# CELL 3: INPUT DATA
# ==========================================================

SP_GR = 0.7593  # SAM

# ==========================================================
# SAMPLE 11 EXPERIMENTAL
# ==========================================================

pressure = np.array([
    590.00,
    702.22,
    814.44,
    926.67,
    1038.89,
    1151.11,
    1263.33,
    1375.56,
    1487.78,
    1600.00
])


# ==========================================================
# EXPERIMENTAL
# ==========================================================

experimental = np.array([
    287.94,
    289.30,
    290.32,
    291.09,
    291.66,
    292.12,
    292.54,
    292.98,
    293.53,
    294.26
])
# ==========================================================
# CELL 4: CREATE INPUT DATASET
# ==========================================================

X_new = pd.DataFrame({

    "Specific Gravity": [SP_GR] * len(pressure),

    "Pressure (psia)": pressure

})

# ==========================================================
# CELL 5: BASE MODEL PREDICTIONS
# ==========================================================

gpr_pred = gpr_model.predict(
    X_new
)

xgb_pred = xgb_model.predict(
    X_new
)

X_mlp = MLP_X_scaler.transform(
    X_new
)

mlp_pred = mlp_model.predict(
    X_mlp
)

if len(mlp_pred.shape) > 1:
    mlp_pred = mlp_pred.ravel()

# ==========================================================
# CELL 6: CREATE META FEATURES
# ==========================================================

meta_features = pd.DataFrame({

    "GPR": gpr_pred,

    "XGB": xgb_pred,

    #"MLP": mlp_pred

})

# ==========================================================
# CELL 7: META MODEL PREDICTIONS
# ==========================================================

hybrid_linear_pred = linear_meta.predict(
    meta_features
)

hybrid_xgb_pred = xgb_meta.predict(
    meta_features
)

# ==========================================================
# CELL 8: METRICS FUNCTION
# ==========================================================

def calculate_metrics(y_true, y_pred):

    r2 = r2_score(
        y_true,
        y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    rrmse = (
        rmse /
        np.mean(y_true)
    ) * 100

    return r2, mae, rmse, rrmse

# ==========================================================
# CELL 9: EXTERNAL VALIDATION
# ==========================================================

gpr_metrics = calculate_metrics(
    experimental,
    gpr_pred
)

xgb_metrics = calculate_metrics(
    experimental,
    xgb_pred
)

mlp_metrics = calculate_metrics(
    experimental,
    mlp_pred
)

linear_metrics = calculate_metrics(
    experimental,
    hybrid_linear_pred
)

xgbmeta_metrics = calculate_metrics(
    experimental,
    hybrid_xgb_pred
)

metrics = pd.DataFrame({

    "Model": [
        "GPR",
        "XGB",
        "MLP",
        "Hybrid_Linear_Meta",
        "Hybrid_XGB_Meta"
    ],

    "R2": [
        gpr_metrics[0],
        xgb_metrics[0],
        mlp_metrics[0],
        linear_metrics[0],
        xgbmeta_metrics[0]
    ],

    "MAE": [
        gpr_metrics[1],
        xgb_metrics[1],
        mlp_metrics[1],
        linear_metrics[1],
        xgbmeta_metrics[1]
    ],

    "RMSE": [
        gpr_metrics[2],
        xgb_metrics[2],
        mlp_metrics[2],
        linear_metrics[2],
        xgbmeta_metrics[2]
    ],

    "RRMSE (%)": [
        gpr_metrics[3],
        xgb_metrics[3],
        mlp_metrics[3],
        linear_metrics[3],
        xgbmeta_metrics[3]
    ]

})

print("\nExternal Validation Results\n")

print(
    metrics.sort_values(
        by="R2",
        ascending=False
    )
)

# ==========================================================
# CELL 10: PREDICTIONS TABLE
# ==========================================================

results = pd.DataFrame({

    "Pressure (psia)": pressure,

    "Experimental": experimental,

    "GPR_Pred": gpr_pred,

    "XGB_Pred": xgb_pred,

    "MLP_Pred": mlp_pred,

    "Hybrid_Linear_Pred": hybrid_linear_pred,

    "Hybrid_XGB_Pred": hybrid_xgb_pred

})

print(results)

results.to_excel(
    "SPGR_Predictions.xlsx",
    index=False
)

print("\nResults Saved Successfully")


External Validation Results

                Model        R2       MAE      RMSE  RRMSE (%)
1                 XGB  0.815822  0.628118  0.796433   0.273150
3  Hybrid_Linear_Meta  0.620931  1.037408  1.142588   0.391869
4     Hybrid_XGB_Meta  0.609352  1.021382  1.159908   0.397809
2                 MLP  0.230542  1.593438  1.627882   0.558308
0                 GPR  0.197033  1.567428  1.662951   0.570336
   Pressure (psia)  Experimental    GPR_Pred    XGB_Pred    MLP_Pred  \
0           590.00        287.94  285.718297  286.539856  285.586255   
1           702.22        289.30  287.956947  287.991791  287.360352   
2           814.44        290.32  288.055144  289.112915  288.785418   
3           926.67        291.09  289.662637  290.194519  289.414417   
4          1038.89        291.66  289.265397  291.329529  289.999555   
5          1151.11        292.12  291.313291  291.833771  290.746592   
6          1263.33        292.54  291.678005  292.647095  291.289603   
7          1375.